# Skillaro: fine-tuning the retrieval models

This notebook reproduces how Skillaro's search was trained. Skillaro's learning assistant answers students
from a curriculum of **1,326 documents split into 25,560 chunks**. Before answering, it has to find the right
chunks. Two models do that:

1. **Embedder** (`BAAI/bge-small-en-v1.5`): turns questions and chunks into 384-number vectors; the closest
   vectors are the best matches. A *bi-encoder*: fast, searches all 25,560 chunks.
2. **Re-ranker** (`cross-encoder/ms-marco-MiniLM-L6-v2`): reads the question and each of the top candidates
   together and reorders them. A *cross-encoder*: slower, more precise, used on the top 10 only.

Both are **fine-tuned** (training continues from the pre-trained model) on 2,997 student-style questions that
a local LLM (Gemma 4) wrote about 999 curriculum chunks. Questions are split **by source document**
(80/10/10), so the test questions come from documents the models never saw.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dilmith-loops/dockerhosting/blob/main/notebooks/skillaro_retrieval_finetuning.ipynb)

**Run on a GPU:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. The full notebook takes
about 25–35 minutes.

| Step | What happens |
|---|---|
| 0 | Install, load the data |
| 1 | Baseline: the original embedder, measured on the test questions |
| 2 | Fine-tune the embedder (contrastive loss with hard negatives) |
| 3 | Fine-tune the re-ranker and compare it with the old rule-based scorer |
| 4 | Results: tables, charts, confidence intervals |
| 5 | Export to ONNX, the format the Skillaro server runs |

## 0. Setup

In [ ]:
!pip -q install "sentence-transformers>=5" datasets onnx onnxruntime

import json, random, time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

# QUICK = True runs a small version end to end in a few minutes (to check the notebook, not for results)
QUICK = False

### Load the data

The data is `skillaro-retrieval-data.zip` (3.9 MB), published next to this notebook. It holds two files:

* `corpus.jsonl`: all 25,560 chunks, each with its source document and its train/valid/test split
* `pairs.jsonl`: the 3 questions Gemma wrote for each of 999 sampled chunks

The cell downloads it from GitHub; a copy in `MyDrive/skillaro/` is used instead if present.

In [ ]:
import os, zipfile
import urllib.request
ZIP = "skillaro-retrieval-data.zip"
URL = "https://raw.githubusercontent.com/dilmith-loops/dockerhosting/main/notebooks/" + ZIP
DRIVE_ZIP = "/content/drive/MyDrive/skillaro/" + ZIP
if os.path.exists(DRIVE_ZIP):          # Drive is mounted (Files panel → Mount Drive) and holds a copy
    ZIP = DRIVE_ZIP
elif not os.path.exists(ZIP):
    urllib.request.urlretrieve(URL, ZIP)
print(f"Using {ZIP} ({os.path.getsize(ZIP) / 2**20:.1f} MB)")
zipfile.ZipFile(ZIP).extractall("data")
print(sorted(os.listdir("data")))

In [ ]:
corpus = [json.loads(line) for line in open("data/corpus.jsonl")]
queries = {"train": [], "valid": [], "test": []}
for line in open("data/pairs.jsonl"):
    p = json.loads(line)
    for q in p["questions"]:
        queries[p["split"]].append({"question": q, "chunk_id": p["chunk_id"], "doc_id": p["doc_id"]})

if QUICK:  # a small slice: the sampled chunks plus 3,000 others; fewer questions
    keep = {q["chunk_id"] for s in queries.values() for q in s}
    keep |= set(random.Random(SEED).sample(range(len(corpus)), 3000))
    remap = {old: new for new, old in enumerate(sorted(keep))}
    corpus = [dict(corpus[old], chunk_id=new) for old, new in sorted(remap.items(), key=lambda x: x[1])]
    queries = {s: [dict(q, chunk_id=remap[q["chunk_id"]]) for q in v[:{"train": 300}.get(s, 90)]] for s, v in queries.items()}

print(f"{len(corpus):,} chunks from {len({c['doc_id'] for c in corpus}):,} documents")
print({s: len(v) for s, v in queries.items()}, "questions (train / valid / test)")
# One question for each of three different test chunks (each chunk has three questions)
examples = [q for i, q in enumerate(queries["test"]) if i % 3 == 0][:3]
pd.set_option("display.max_colwidth", 120)
pd.DataFrame([{"student question (written by Gemma)": q["question"],
               "chunk it was written from": corpus[q["chunk_id"]]["content"][:220].replace("\n", " ") + "…"}
              for q in examples])

## 1. Baseline: how good is the original embedder?

Each test question searches **all** chunks. Only the chunk the question was written from counts as correct
(strict: a neighbouring chunk that also answers it still counts as a miss). Metrics:

* **Recall@k**: share of questions whose correct chunk is in the top *k*
* **MRR@10**: mean of 1 / rank of the correct chunk (0 if not in the top 10); rewards ranking it first

In [ ]:
from sentence_transformers import SentenceTransformer

BASE_EMBEDDER = "BAAI/bge-small-en-v1.5"
BASE_RERANKER = "cross-encoder/ms-marco-MiniLM-L6-v2"
MAX_LENGTH = 256          # 600-character chunks are about 150 tokens

def embed_corpus(model):
    return model.encode([c["content"] for c in corpus], batch_size=32, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=True)

def rank(model, corpus_emb, qs, k=100):
    q = model.encode([x["question"] for x in qs], batch_size=32, normalize_embeddings=True, convert_to_numpy=True)
    sims = q @ corpus_emb.T
    top = np.argsort(-sims, axis=1)[:, :k]
    return top, np.take_along_axis(sims, top, axis=1)

def positions(rankings, qs):
    pos = []
    for ids, q in zip(rankings, qs):
        hit = np.where(np.asarray(ids) == q["chunk_id"])[0]
        pos.append(int(hit[0]) + 1 if len(hit) else 10**6)
    return np.asarray(pos)

def metrics(rankings, qs):
    pos = positions(rankings, qs)
    return {**{f"recall@{k}": round(float(np.mean(pos <= k)), 3) for k in (1, 3, 10)},
            "mrr@10": round(float(np.mean(np.where(pos <= 10, 1.0 / pos, 0.0))), 3)}

def reciprocal_ranks(rankings, qs):
    pos = positions(rankings, qs)
    return np.where(pos <= 10, 1.0 / pos, 0.0)

def free_gpu():
    """Releases cached GPU memory between stages (models from earlier stages are deleted first)."""
    import gc; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    elif DEVICE == "mps": torch.mps.empty_cache()

base = SentenceTransformer(BASE_EMBEDDER, device=DEVICE)
base.max_seq_length = MAX_LENGTH
t = time.time()
base_corpus = embed_corpus(base)
print(f"Embedded {len(corpus):,} chunks in {time.time() - t:.0f} s → matrix {base_corpus.shape}")

results = {}
for split in ("valid", "test"):
    top, _ = rank(base, base_corpus, queries[split])
    results[("Original bge-small", split)] = metrics(top, queries[split])
pd.DataFrame(results).T

### Hard negatives

A *hard negative* is a passage the original model ranks high for a question but that is wrong. Training
on them teaches the model the fine distinctions. Negatives come only from **other documents**: neighbouring
chunks of the same document overlap the right answer and often answer it too.

In [ ]:
def mine_negatives(top, qs, per_query):
    return [[int(i) for i in ids if corpus[i]["doc_id"] != q["doc_id"]][:per_query] for ids, q in zip(top, qs)]

top, _ = rank(base, base_corpus, queries["train"], k=50)
negatives = mine_negatives(top, queries["train"], per_query=1)
top_base, _ = rank(base, base_corpus, queries["test"], k=10)   # kept for the significance test in step 4
del base, base_corpus; free_gpu()
train_rows = [(q, n[0]) for q, n in zip(queries["train"], negatives) if n]
q, neg = train_rows[0]
print("Question:        ", q["question"])
print("Right passage:   ", corpus[q["chunk_id"]]["content"][:200], "…")
print("Hard negative:   ", corpus[neg]["content"][:200], "…")

## 2. Fine-tune the embedder

**Loss: multiple-negatives ranking (contrastive).** In each batch of 32, every question must score its own
passage above the 63 other passages in the batch (32 right passages + 32 hard negatives). The *cached*
version gives the same result while processing the batch in smaller pieces, so it fits in memory.

Settings: 2 epochs, learning rate 2e-5, batch 32, 10% warm-up. A checkpoint is saved after each epoch and
the better one on the validation questions is kept.

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

train_data = Dataset.from_dict({
    "anchor":   [q["question"] for q, _ in train_rows],
    "positive": [corpus[q["chunk_id"]]["content"] for q, _ in train_rows],
    "negative": [corpus[n]["content"] for _, n in train_rows],
})
torch.manual_seed(SEED)
# Eager attention on Apple GPUs (the cached loss needs dropout, which MPS's fused attention lacks)
kwargs = {"attn_implementation": "eager"} if DEVICE == "mps" else {}
embedder = SentenceTransformer(BASE_EMBEDDER, device=DEVICE, model_kwargs=kwargs)
embedder.max_seq_length = MAX_LENGTH
args = SentenceTransformerTrainingArguments(
    output_dir="embedder-checkpoints", num_train_epochs=1 if QUICK else 2, learning_rate=2e-5, warmup_ratio=0.1,
    per_device_train_batch_size=32, batch_sampler=BatchSamplers.NO_DUPLICATES, save_strategy="epoch",
    logging_steps=5, seed=SEED, report_to=[], fp16=DEVICE == "cuda")
trainer = SentenceTransformerTrainer(model=embedder, args=args, train_dataset=train_data,
                                     loss=CachedMultipleNegativesRankingLoss(embedder, mini_batch_size=8))
t = time.time()
trainer.train()
print(f"Trained on {len(train_rows):,} triplets in {(time.time() - t) / 60:.1f} min")

log = [x for x in trainer.state.log_history if "loss" in x]
del trainer, embedder; free_gpu()
plt.figure(figsize=(7, 3))
plt.plot([x["epoch"] for x in log], [x["loss"] for x in log], marker="o", ms=3)
plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("Embedder fine-tuning"); plt.grid(alpha=.3)
plt.show()

In [ ]:
import glob
epochs, best = {}, None
for ckpt in sorted(glob.glob("embedder-checkpoints/checkpoint-*"), key=lambda p: int(p.split("-")[-1])):
    m = SentenceTransformer(ckpt, device=DEVICE); m.max_seq_length = MAX_LENGTH
    emb = embed_corpus(m)
    top, _ = rank(m, emb, queries["valid"])
    epochs[ckpt] = metrics(top, queries["valid"])
    print(ckpt, epochs[ckpt])
    if best is None or epochs[ckpt]["mrr@10"] > epochs[best[0]]["mrr@10"]:
        best = (ckpt, m, emb)          # only the best model so far stays in memory
    del m; free_gpu()
best_ckpt, embedder, tuned_corpus = best
del best; free_gpu()
print("Kept:", best_ckpt)

for split in ("valid", "test"):
    top, _ = rank(embedder, tuned_corpus, queries[split])
    results[("Fine-tuned bge-small", split)] = metrics(top, queries[split])
pd.DataFrame(results).T

## 3. Fine-tune the re-ranker

The re-ranker reorders the fine-tuned embedder's **top 10**. It is trained on (question, passage) pairs:
the right passage labelled 1 and 3 hard negatives (from the fine-tuned embedder's results) labelled 0, with
binary cross-entropy, for 1 epoch.

It is compared with three alternatives on the same top-10 lists: no re-ranking, the **rule-based scorer**
Skillaro used before (word overlap, title match and similarity, copied below), and the re-ranker before
fine-tuning.

In [ ]:
import re

def rule_based_rerank(query, candidates):
    """Skillaro's previous re-ranker (services/learning-service/app/services/reranker.py)."""
    tokens = [w.lower() for w in re.findall(r"\b\w+\b", query) if len(w) > 2]
    q = query.lower().strip()
    scored = []
    for c in candidates:
        content, title, topic = c["content"].lower(), c["title"].lower(), c["topic"].lower()
        phrase = 0.25 if q in content or q in title else 0.0
        ratio = sum(t in content for t in tokens) / max(len(tokens), 1)
        affinity = sum(t in title or t in topic for t in tokens) / max(len(tokens), 1) * 0.2
        score = c["similarity"] * 0.45 + ratio * 0.30 + phrase + affinity
        scored.append((min(1.0, max(0.0, score)), c["chunk_id"]))
    return [cid for _, cid in sorted(scored, key=lambda x: -x[0])]

def candidates(qs):
    top, sims = rank(embedder, tuned_corpus, qs, k=10)
    return [[{"chunk_id": int(i), "content": corpus[i]["content"], "title": corpus[i]["title"],
              "topic": corpus[i]["topic"], "similarity": float(s)} for i, s in zip(ids, ss)]
            for ids, ss in zip(top, sims)]

cands = {s: candidates(queries[s]) for s in ("valid", "test")}
for split in ("valid", "test"):
    results[("+ no re-ranking", split)] = metrics([[c["chunk_id"] for c in cs] for cs in cands[split]], queries[split])
    results[("+ rule-based re-ranker (before)", split)] = metrics(
        [rule_based_rerank(q["question"], cs) for q, cs in zip(queries[split], cands[split])], queries[split])
pd.DataFrame(results).T

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

top, _ = rank(embedder, tuned_corpus, queries["train"], k=30)
q_, p_, y_ = [], [], []
for q, negs in zip(queries["train"], mine_negatives(top, queries["train"], per_query=3)):
    q_.append(q["question"]); p_.append(corpus[q["chunk_id"]]["content"]); y_.append(1.0)
    for n in negs:
        q_.append(q["question"]); p_.append(corpus[n]["content"]); y_.append(0.0)
rerank_data = Dataset.from_dict({"query": q_, "passage": p_, "label": y_})
print(f"{len(y_):,} training pairs ({int(sum(y_)):,} right, {len(y_) - int(sum(y_)):,} wrong)")

def ce_rank(model, split):
    out = []
    for q, cs in zip(queries[split], cands[split]):
        s = model.predict([(q["question"], c["content"]) for c in cs], batch_size=32, show_progress_bar=False)
        out.append([cs[i]["chunk_id"] for i in np.argsort(-np.asarray(s))])
    return out

torch.manual_seed(SEED)
reranker = CrossEncoder(BASE_RERANKER, num_labels=1, max_length=MAX_LENGTH, device=DEVICE)
for split in ("valid", "test"):
    results[("+ MiniLM re-ranker, not fine-tuned", split)] = metrics(ce_rank(reranker, split), queries[split])

args = CrossEncoderTrainingArguments(
    output_dir="reranker-checkpoints", num_train_epochs=1, learning_rate=2e-5, warmup_ratio=0.1,
    per_device_train_batch_size=32, save_strategy="no", logging_steps=10, seed=SEED, report_to=[],
    fp16=DEVICE == "cuda")
ce_trainer = CrossEncoderTrainer(model=reranker, args=args, train_dataset=rerank_data, loss=BinaryCrossEntropyLoss(reranker))
t = time.time()
ce_trainer.train()
print(f"Trained in {(time.time() - t) / 60:.1f} min")
for split in ("valid", "test"):
    results[("+ fine-tuned re-ranker", split)] = metrics(ce_rank(reranker, split), queries[split])

log = [x for x in ce_trainer.state.log_history if "loss" in x]
del ce_trainer; free_gpu()
plt.figure(figsize=(7, 3))
plt.plot([x["epoch"] for x in log], [x["loss"] for x in log], marker="o", ms=3, color="tab:purple")
plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("Re-ranker fine-tuning"); plt.grid(alpha=.3)
plt.show()

## 4. Results

All numbers are on questions about documents the models never saw. Re-ranking only reorders the top 10, so
it can't change recall@10.

In [ ]:
table = pd.DataFrame(results).T
table.index = pd.MultiIndex.from_tuples(table.index, names=["method", "split"])
display(table.xs("test", level="split").style.format("{:.3f}").highlight_max(color="#d0ffa4"))

test = table.xs("test", level="split")
order = ["Original bge-small", "Fine-tuned bge-small", "+ rule-based re-ranker (before)", "+ fine-tuned re-ranker"]
colors = ["#9aa0a6", "#006ee9", "#eba0ff", "#2e7d32"]
plt.figure(figsize=(8, 3.6))
bars = plt.barh(order[::-1], test.loc[order[::-1], "mrr@10"], color=colors[::-1])
plt.bar_label(bars, fmt="%.3f", padding=3)
plt.xlim(0, 1); plt.xlabel("MRR@10 on test (higher is better)"); plt.title("Search quality before and after fine-tuning")
plt.tight_layout(); plt.show()

### Is the gain real? Paired bootstrap

The same test questions are resampled 10,000 times; if the 95% interval of the improvement stays above zero,
the gain isn't luck from this particular set of questions.

In [ ]:
def bootstrap(a, b, n=10_000):
    d = np.asarray(a) - np.asarray(b)
    means = d[np.random.default_rng(SEED).integers(0, len(d), (n, len(d)))].mean(1)
    return np.percentile(means, [2.5, 97.5]).round(3)

top_tuned, _ = rank(embedder, tuned_corpus, queries["test"], k=10)   # top_base was saved in step 1
rr_base, rr_tuned = reciprocal_ranks(top_base, queries["test"]), reciprocal_ranks(top_tuned, queries["test"])
rr_rerank = reciprocal_ranks(ce_rank(reranker, "test"), queries["test"])
print("Fine-tuned embedder vs original, MRR@10 gain 95% CI:", bootstrap(rr_tuned, rr_base))
print("Fine-tuned re-ranker vs no re-ranking, MRR@10 gain 95% CI:", bootstrap(rr_rerank, rr_tuned))

## 5. Export to ONNX

The Skillaro server runs the models with **onnxruntime** on CPU (no PyTorch). The embedder is exported in
full precision (int8 cost 0.012 MRR with no speed gain); the re-ranker with 8-bit weights (22 MB). The
files are saved to `skillaro-models/` and, on Colab, copied to your Drive.

In [ ]:
import os, shutil
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

os.makedirs("skillaro-models/embedder", exist_ok=True); os.makedirs("skillaro-models/reranker", exist_ok=True)
names = ["input_ids", "attention_mask", "token_type_ids"]

class Encoder(torch.nn.Module):          # last hidden state; the service takes the CLS vector and normalises
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask, token_type_ids):
        return self.m(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids).last_hidden_state

hf = embedder[0].auto_model.eval().cpu()
enc = embedder.tokenizer(["a question", "a longer passage about indexes"], padding=True, return_tensors="pt")
torch.onnx.export(Encoder(hf), tuple(enc[n] for n in names), "skillaro-models/embedder/model.onnx",
                  input_names=names, output_names=["last_hidden_state"], opset_version=17, dynamo=False,
                  dynamic_axes={**{n: {0: "batch", 1: "seq"} for n in names}, "last_hidden_state": {0: "batch", 1: "seq"}})
embedder.tokenizer.backend_tokenizer.save("skillaro-models/embedder/tokenizer.json")

ce = reranker.model.eval().cpu()
enc = reranker.tokenizer(["q", "q"], ["a passage", "another passage"], padding=True, return_tensors="pt")
torch.onnx.export(ce, tuple(enc[n] for n in names), "skillaro-models/reranker/model.fp32.onnx", input_names=names,
                  output_names=["logits"], opset_version=17, dynamo=False,
                  dynamic_axes={**{n: {0: "batch", 1: "seq"} for n in names}, "logits": {0: "batch"}})
quantize_dynamic("skillaro-models/reranker/model.fp32.onnx", "skillaro-models/reranker/model.onnx", weight_type=QuantType.QInt8)
os.remove("skillaro-models/reranker/model.fp32.onnx")
reranker.tokenizer.backend_tokenizer.save("skillaro-models/reranker/tokenizer.json")

# Check: the exported embedder gives the same vectors as the trained model
sess = ort.InferenceSession("skillaro-models/embedder/model.onnx", providers=["CPUExecutionProvider"])
text = corpus[0]["content"]
e = embedder.tokenizer([text], return_tensors="np", truncation=True, max_length=512)
cls = sess.run(None, {n: e[n].astype(np.int64) for n in names})[0][:, 0]
onnx_vec = cls / np.linalg.norm(cls, axis=1, keepdims=True)
embedder.to(DEVICE)
torch_vec = embedder.encode([text], normalize_embeddings=True)
print("cosine(ONNX, PyTorch) =", float((onnx_vec @ torch_vec.T)[0, 0]))
for f in ("embedder/model.onnx", "reranker/model.onnx"):
    print(f, f"{os.path.getsize('skillaro-models/' + f) / 2**20:.1f} MB")

if os.path.isdir("/content/drive/MyDrive"):  # only if Drive is mounted
    shutil.copytree("skillaro-models", "/content/drive/MyDrive/skillaro/skillaro-models", dirs_exist_ok=True)
    print("Copied to Drive: MyDrive/skillaro/skillaro-models")
else:
    print("Saved in skillaro-models/ (Files panel); mount Drive to keep a copy")

## Summary

* **Embedder** (bge-small, 33M parameters, deep learning bi-encoder): contrastive fine-tuning with hard
  negatives, 2 epochs (best on validation kept).
* **Re-ranker** (MiniLM cross-encoder, 22M parameters): binary cross-entropy, 1 epoch; it replaces a
  rule-based scorer that ranked worse than no re-ranking.
* Measured on questions about **unseen documents**, over all 25,560 chunks, with bootstrap confidence intervals.

Recorded run (Apple M-series GPU, 18–19 October 2026; Colab runs differ slightly):

| Test (339 questions) | recall@1 | recall@3 | recall@10 | MRR@10 |
|---|---|---|---|---|
| Original bge-small | 0.325 | 0.513 | 0.679 | 0.437 |
| Fine-tuned bge-small | 0.437 | 0.625 | 0.758 | 0.544 |
| + rule-based re-ranker (before) | 0.404 | 0.581 | 0.743 | 0.512 |
| + fine-tuned re-ranker | 0.472 | 0.673 | 0.743 | 0.573 |

The production pipeline also fuses the embedder's results with PostgreSQL keyword search (Reciprocal Rank
Fusion) before re-ranking; this notebook evaluates the two learned models on their own.